<a href="https://colab.research.google.com/github/RobertFlan02/Dietetics-FYP/blob/main/Dietetics-FYP/Experiments/YOLO/Experiment%202/YOLO2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
#!/usr/bin/env python
# coding: utf-8

# ------------------------------
# 1. Imports and Configuration
# ------------------------------
import os
import io
import cv2
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
import shutil
import yaml

# Paths to parquet files
TRAIN_PARQUET_PATHS = [
    "/content/drive/MyDrive/foodseg103/train-00000-of-00003-6bb37ec387d1825a.parquet",
    "/content/drive/MyDrive/foodseg103/train-00001-of-00003-4a1caa37147c0681.parquet",
    "/content/drive/MyDrive/foodseg103/train-00002-of-00003-c8b698399244cd95.parquet"
]

VAL_PARQUET_PATHS = [
    "/content/drive/MyDrive/foodseg103/validation-00000-of-00001-a5bfdaa5beb7006a.parquet"
]

# Output directories for YOLO training data
OUTPUT_DIR = "/content/drive/MyDrive/food_yolo"
TRAIN_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images/train")
TRAIN_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels/train")
VAL_IMAGES_DIR = os.path.join(OUTPUT_DIR, "images/val")
VAL_LABELS_DIR = os.path.join(OUTPUT_DIR, "labels/val")

# Padding (in pixels) to add around bounding boxes
PADDING = 10
# Background class value (assumed to be 0)
BACKGROUND_CLASS = 0

# ------------------------------
# 2. Helper Functions for Dataset Conversion
# ------------------------------
def load_parquet_files(parquet_paths):
    """Load and concatenate parquet files into a single DataFrame."""
    dfs = []
    for path in parquet_paths:
        try:
            df = pd.read_parquet(path)
            dfs.append(df)
        except Exception as e:
            print(f"Error loading {path}: {e}")
    if dfs:
        full_df = pd.concat(dfs, ignore_index=True)
        return full_df
    else:
        return pd.DataFrame()

def extract_image_and_mask(row):
    """
    Extracts an image and its corresponding mask from a DataFrame row.
    Expects row['image'] and row['label'] to be dictionaries with key "bytes".
    """
    img_bytes = row['image'].get("bytes")
    if img_bytes is None:
        raise ValueError("No image bytes found.")
    img = Image.open(io.BytesIO(img_bytes)).convert("RGB")

    mask_bytes = row['label'].get("bytes")
    if mask_bytes is None:
        raise ValueError("No mask bytes found.")
    mask = Image.open(io.BytesIO(mask_bytes))
    return img, mask

def compute_bounding_boxes(mask):
    """
    Compute bounding boxes for each connected component in the mask.
    Returns a list of tuples: (class_id, x, y, w, h).
    """
    mask_np = np.array(mask)
    boxes = []
    for cls in np.unique(mask_np):
        if cls == BACKGROUND_CLASS:
            continue
        binary = np.uint8(mask_np == cls) * 255
        contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        for cnt in contours:
            if cv2.contourArea(cnt) < 10:
                continue
            x, y, w, h = cv2.boundingRect(cnt)
            boxes.append((int(cls), x, y, w, h))
    return boxes

def convert_bbox_to_yolo(box, img_width, img_height, padding=PADDING):
    cls, x, y, w, h = box
    new_cls = cls - 1  # shift food labels down by 1
    x_padded = max(0, x - padding)
    y_padded = max(0, y - padding)
    x2_padded = min(img_width, x + w + padding)
    y2_padded = min(img_height, y + h + padding)
    w_padded = x2_padded - x_padded
    h_padded = y2_padded - y_padded
    x_center = (x_padded + w_padded / 2) / img_width
    y_center = (y_padded + h_padded / 2) / img_height
    w_norm = w_padded / img_width
    h_norm = h_padded / img_height
    return new_cls, x_center, y_center, w_norm, h_norm

def save_yolo_annotation(image_id, boxes, dest_folder):
    """
    Save YOLO formatted annotation file for a given image.
    Each line: <class> <x_center> <y_center> <width> <height>
    """
    txt_path = os.path.join(dest_folder, f"{image_id}.txt")
    with open(txt_path, "w") as f:
        for box in boxes:
            cls, x_center, y_center, w_norm, h_norm = box
            f.write(f"{cls} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

def process_parquet_to_yolo(parquet_paths, images_dest, labels_dest):
    """Process a set of parquet files to create YOLO images and label files."""
    df = load_parquet_files(parquet_paths)
    if df.empty:
        print("No data loaded from the given parquet files.")
        return
    os.makedirs(images_dest, exist_ok=True)
    os.makedirs(labels_dest, exist_ok=True)
    print(f"Processing {len(df)} samples...")
    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Converting data"):
        try:
            img, mask = extract_image_and_mask(row)
            image_id = str(row.get("id", idx))
            img_save_path = os.path.join(images_dest, f"{image_id}.jpg")
            img.save(img_save_path, format="JPEG")
            img_width, img_height = img.size
            raw_boxes = compute_bounding_boxes(mask)
            if not raw_boxes:
                open(os.path.join(labels_dest, f"{image_id}.txt"), "w").close()
                continue
            yolo_boxes = [convert_bbox_to_yolo(box, img_width, img_height, padding=PADDING)
                          for box in raw_boxes]
            save_yolo_annotation(image_id, yolo_boxes, labels_dest)
        except Exception as e:
            print(f"Error processing sample {idx}: {e}")

# ------------------------------
# 3. Process Training and Validation Data
# ------------------------------
print("Processing training data...")
process_parquet_to_yolo(TRAIN_PARQUET_PATHS, TRAIN_IMAGES_DIR, TRAIN_LABELS_DIR)
print("Processing validation data...")
process_parquet_to_yolo(VAL_PARQUET_PATHS, VAL_IMAGES_DIR, VAL_LABELS_DIR)
print("Conversion complete.")

# ------------------------------
# 4. Oversample Rare Classes in the Training Set
# ------------------------------
# First, read through the training label files and compute class frequencies.
label_files = [os.path.join(TRAIN_LABELS_DIR, f) for f in os.listdir(TRAIN_LABELS_DIR) if f.endswith('.txt')]
class_counts = {}
image_classes = {}  # Map each label file to the list of classes in that image.

for label_file in label_files:
    with open(label_file, 'r') as f:
        lines = f.readlines()
    classes_in_image = []
    for line in lines:
        parts = line.strip().split()
        if len(parts) < 5:
            continue
        cls = int(parts[0])
        classes_in_image.append(cls)
        class_counts[cls] = class_counts.get(cls, 0) + 1
    image_classes[label_file] = classes_in_image

print("Class frequencies in training set:", class_counts)

# Define rarity threshold (for example, any class with fewer than 50 instances is rare)
rare_threshold = 50
rare_classes = [cls for cls, count in class_counts.items() if count < rare_threshold]
print("Rare classes identified:", rare_classes)

# Duplicate images containing any rare class to oversample them.
oversample_factor = 3  # Number of extra copies to make per image with a rare class.
dup_count = 0

for label_file, classes in image_classes.items():
    if any(cls in rare_classes for cls in classes):
        base = os.path.splitext(os.path.basename(label_file))[0]
        image_file = os.path.join(TRAIN_IMAGES_DIR, base + ".jpg")
        for i in range(oversample_factor):
            new_base = f"{base}_dup{i}"
            new_image_file = os.path.join(TRAIN_IMAGES_DIR, new_base + ".jpg")
            new_label_file = os.path.join(TRAIN_LABELS_DIR, new_base + ".txt")
            shutil.copy(image_file, new_image_file)
            shutil.copy(label_file, new_label_file)
            dup_count += 1

print(f"Duplicated {dup_count} images for oversampling rare classes.")

# ------------------------------
# 5. Generate YOLO YAML Configuration File
# ------------------------------
csv_path = "/content/drive/MyDrive/foodseg103/class_mappings.csv"
df = pd.read_csv(csv_path)
df = df[df["Class Id"] > 0]
names = {}
for _, row in df.iterrows():
    new_id = int(row["Class Id"]) - 1
    names[new_id] = row["Class Name"]

dataset_yaml = {
    "path": OUTPUT_DIR,
    "train": "images/train",
    "val": "images/val",
    "names": names
}

yaml_path = os.path.join(OUTPUT_DIR, "food_yolo.yaml")
with open(yaml_path, 'w') as f:
    yaml.dump(dataset_yaml, f, default_flow_style=False)
print(f"YAML file saved to: {yaml_path}")

print("Dataset preparation and oversampling complete. You can now train your YOLO model using the prepared dataset.")


Processing training data...
Processing 4983 samples...


Converting data: 100%|██████████| 4983/4983 [44:06<00:00,  1.88it/s]


Processing validation data...
Processing 2135 samples...


Converting data: 100%|██████████| 2135/2135 [18:12<00:00,  1.95it/s]


Conversion complete.
Class frequencies in training set: {51: 1066, 57: 1616, 72: 1374, 93: 528, 2: 189, 55: 202, 65: 629, 79: 710, 47: 1111, 58: 401, 40: 185, 56: 78, 23: 305, 27: 96, 49: 207, 53: 324, 46: 610, 64: 298, 95: 337, 9: 475, 30: 463, 71: 233, 83: 1379, 88: 1021, 100: 190, 66: 490, 69: 999, 84: 312, 90: 138, 82: 55, 20: 38, 45: 939, 76: 58, 7: 854, 36: 608, 63: 155, 28: 156, 29: 718, 8: 436, 81: 547, 94: 232, 86: 942, 101: 12, 24: 189, 48: 364, 92: 695, 70: 140, 10: 117, 60: 71, 87: 233, 50: 83, 31: 554, 32: 107, 43: 277, 67: 105, 75: 280, 98: 9, 80: 108, 102: 217, 4: 345, 3: 151, 15: 29, 39: 195, 13: 151, 77: 25, 89: 96, 99: 11, 62: 10, 37: 51, 12: 136, 11: 99, 16: 266, 42: 39, 34: 93, 21: 99, 68: 33, 97: 182, 74: 16, 91: 33, 61: 22, 38: 51, 44: 65, 35: 132, 0: 56, 26: 39, 19: 79, 78: 33, 33: 75, 14: 47, 54: 73, 85: 8, 5: 34, 18: 43, 22: 16, 17: 44, 52: 19, 25: 14, 96: 12, 41: 64, 73: 4, 59: 7, 1: 8, 6: 5}
Rare classes identified: [20, 101, 98, 15, 77, 99, 62, 42, 68, 74, 9

In [ ]:
import pandas as pd
import yaml

csv_path = "/content/drive/MyDrive/foodseg103/class_mappings.csv"
df = pd.read_csv(csv_path)

# Only include rows where Class Id > 0 (i.e., ignore background)
df = df[df["Class Id"] > 0]

# Create names dictionary by re-indexing: new_id = original_id - 1
names = {}
for _, row in df.iterrows():
    new_id = int(row["Class Id"]) - 1  # shift food labels down by 1
    names[new_id] = row["Class Name"]

dataset_yaml = {
    "path": "/content/drive/MyDrive/food_yolo",
    "train": "images/train",
    "val": "images/val",
    "names": names
}

yaml_path = "/content/drive/MyDrive/food_yolo/food_yolo.yaml"
with open(yaml_path, 'w') as file:
    yaml.dump(dataset_yaml, file, default_flow_style=False)

print(f"YAML file saved to: {yaml_path}")


YAML file saved to: /content/drive/MyDrive/food_yolo/food_yolo.yaml


In [1]:
!pip install ultralytics --upgrade

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 949.8/949.8 kB 26.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 107.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 89.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 56.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 39.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.8 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:
!find /usr/local/lib/python3.11/dist-packages/ultralytics -name loss.py


/usr/local/lib/python3.11/dist-packages/ultralytics/models/utils/loss.py
/usr/local/lib/python3.11/dist-packages/ultralytics/utils/loss.py


In [ ]:
!yolo task=detect mode=train model=yolov8l.pt \
    data=/content/drive/MyDrive/food_yolo/food_yolo.yaml \
    epochs=70 imgsz=640 batch=8 \
    multi_scale=True mosaic=1.0 mixup=0.1 auto_augment=randaugment \
    project=/content/drive/MyDrive/food_yolo/results \
    name=yolov8l_detect_balanced


100% 83.7M/83.7M [00:00<00:00, 441MB/s]
Ultralytics 8.3.95 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (NVIDIA L4, 22693MiB)
engine/trainer: task=detect, mode=train, model=yolov8l.pt, data=/content/drive/MyDrive/food_yolo/food_yolo.yaml, epochs=70, time=None, patience=100, batch=8, imgsz=640, save=True, save_period=-1, cache=False, device=None, workers=8, project=/content/drive/MyDrive/food_yolo/results, name=yolov8l_detect_balanced, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=True, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, vid_stride=1, stream_buffer=False, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, embed=None, show=False,

In [2]:
!yolo task=detect mode=val model=/content/drive/MyDrive/food_yolo/results/yolov8l_detect_balanced/weights/best.pt \
    data=/content/drive/MyDrive/food_yolo/food_yolo.yaml imgsz=640


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.96 🚀 Python-3.11.11 torch-2.6.0+cu124 CUDA:0 (Tesla T4, 15095MiB)
Model summary (fused): 112 layers, 43,686,021 parameters, 0 gradients, 165.3 GFLOPs
100% 755k/755k [00:00<00:00, 24.2MB/s]
val: Scanning /content/drive/MyDrive/food_yolo/labels/val.cache... 2135 images, 0 backgrounds, 0 corrupt: 100% 2135/2135 [00:00<?, ?it/s]
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% 134/134 [01:37<00:00,  1.37it/s]
                   all       2135      11995      0.548       0.36      0.387      0.329
                 candy         11         43      0.558     0.0886        0.2      0.176
              egg tart          1       